In [2]:
import numpy as np

def als_recommendation_system(R, num_factors, num_iterations, reg_param):
    m, n = R.shape  # m = number of users, n = number of items
    U = np.random.rand(m, num_factors)
    I = np.random.rand(n, num_factors)
    known_ratings = np.where(R > 0)
    user_indices, item_indices = known_ratings

    for _ in range(num_iterations):
        for user_id in range(m):
            rated_item_indices = item_indices[user_indices == user_id]
            if len(rated_item_indices) > 0:
                I_subset = I[rated_item_indices, :]
                R_subset = R[user_id, rated_item_indices]
                A = I_subset.T @ I_subset + reg_param * np.eye(num_factors)
                b = I_subset.T @ R_subset
                U[user_id, :] = np.linalg.solve(A, b)

        for item_id in range(n):
            rated_user_indices = user_indices[item_indices == item_id]
            if len(rated_user_indices) > 0:
                U_subset = U[rated_user_indices, :]
                R_subset = R[rated_user_indices, item_id]
                A = U_subset.T @ U_subset + reg_param * np.eye(num_factors)
                b = U_subset.T @ R_subset
                I[item_id, :] = np.linalg.solve(A, b)

    return U, I

def make_predictions(U, I, R):
    predictions = U @ I.T
    predictions[R > 0] = R[R > 0]
    return predictions

if __name__ == '__main__':
    R = np.array([
        [5, 3, 0, 1],
        [4, 0, 0, 1],
        [1, 1, 0, 5],
        [0, 0, 0, 4],
        [0, 1, 5, 4],
    ])

    num_factors = 2       # Number of latent features
    num_iterations = 100  # Number of ALS iterations
    reg_param = 0.1       # Regularization parameter

    user_matrix, item_matrix = als_recommendation_system(R, num_factors, num_iterations, reg_param)
    predicted_ratings = make_predictions(user_matrix, item_matrix, R)

    print("Original Ratings Matrix (R):")
    print(R)
    print("\nPredicted Ratings Matrix:")
    print(np.round(predicted_ratings, 2))

Original Ratings Matrix (R):
[[5 3 0 1]
 [4 0 0 1]
 [1 1 0 5]
 [0 0 0 4]
 [0 1 5 4]]

Predicted Ratings Matrix:
[[5.   3.   1.81 1.  ]
 [4.   2.35 1.66 1.  ]
 [1.   1.   5.79 5.  ]
 [0.66 0.7  4.61 4.  ]
 [1.19 1.   5.   4.  ]]
